<a href="https://colab.research.google.com/github/Mondin0/data-eng/blob/main/CeL_Orquestacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Orquestación

In [ ]:
!pip install -U prefect
# Una vez instalado, reiniciar: Entorno de ejecución > Reiniciar sesión

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 384.9/384.9 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/

Prefect cuenta con una versión open source para alojarlo y ejecutarlo en tu propia computadora, también ofrece una versión cloud paga pero con una capa gratuita.

En esta ocasión, utilizaremos la versión cloud. Este es el link de la plataforma https://app.prefect.cloud/auth/sign-up. Es posible registrarse directamente con una cuenta de gmail.

Una vez preparada la plataforma, es necesario obtener una api_key para ingresar en el siguiente comando

`prefect cloud logik -k <reemplazar con tu api key>`

In [ ]:
!prefect cloud login -k pnu_LEVXy8CprhGW5sXwucr1kHyWQ7mPD2169Qoz

Authenticated with Prefect Cloud! Using workspace 'myaccount/default'.


Cabe aclarar, que para este tipo de desarrollos, no es conveniente utilizar Jupyter Notebook (ya sea en Google Colab o de forma local), sino que se debe utilizar algún IDE o editor de código (como VS Code).

A fines didácticos y para evitar tanta configuración en un entorno local, trabajamos sobre Google Colab con algunas modificaciones.

Varias celdas cuentan al comienzo con un comando que `writefile <nombre script>.py`. Esto es para guardar toda la celda como  un script de Python que luego ejecutaremos.

In [ ]:
!mkdir -p data
!mkdir -p scripts

## ETL básico

In [ ]:
%%writefile scripts/etl_simple.py

import requests
import pandas as pd
from prefect import task, flow
from prefect.runtime import flow_run

BASE_URL = "https://api.luchtmeetnet.nl/open_api"

@task(
    retries=3,
    retry_delay_seconds=5
)
def extract_data():
    organisation_url = f"{BASE_URL}/organizations"
    response = requests.get(organisation_url)
    response.raise_for_status()
    organisations = response.json()["data"]
    return organisations

@task
def transform_data(organisations: list):
    df = pd.json_normalize(organisations)
    return df

@task
def load_data(df: pd.DataFrame):
    scheduled_run = flow_run.scheduled_start_time.strftime("%Y-%m-%dT%H:%M")
    df.to_csv(f"data/organisations_{scheduled_run}.csv")

@flow
def etl_simple():
    organisations = extract_data()
    df = transform_data(organisations)
    load_data(df)

if __name__ == "__main__":
    etl_simple.serve(
        name="ETL-Simple-con-error2",
        cron="*/5 * * * *"
        )

Overwriting scripts/etl_simple.py


In [ ]:
!python scripts/etl_simple.py

Your flow 'etl-simple' is being served and polling for scheduled runs!

To trigger a run for this flow, use the following command:

        $ prefect deployment run 'etl-simple/ETL-Simple-con-error2'

You can also run your flow via the Prefect UI: https://app.prefect.cloud/account/4f464ca9-2008-4694-bae1-d3dad8f2df02/workspace/666ec2f7-4658-47d3-9d17-ebdeb0399ca6/deployments/deployment/32d808d3-2cbe-472c-960e-c2a291b6eef7

00:22:09.421 | INFO    | prefect.flow_runs.runner - Runner 'ETL-Simple-con-error2' submitting flow run '7c16c1fa-039e-4f92-93ea-822ef41aa67e'
00:22:09.582 | INFO    | prefect.flow_runs.runner - Opening process...
00:22:09.599 | INFO    | prefect.flow_runs.runner - Completed submission of flow run '7c16c1fa-039e-4f92-93ea-822ef41aa67e'
00:22:12.422 | INFO    | Flow run 'silky-lizard' - Downloading flow code from storage at '.'
00:22:13.036 | INFO    | Flow run 'silky-lizard' - Beginning flow run 'silky-lizard' for flow 'etl-simple'
00:22:13.038 | INFO    | Flow run 's

## ETL Parametrizable
Se parametrizan los endpoints a consultar

In [ ]:
%%writefile scripts/etl_parametrizable.py

import os
import requests
import pandas as pd
from prefect import task, flow
from prefect.runtime import flow_run

BASE_URL = "https://api.luchtmeetnet.nl/open_api"

@task(
    retries=3,
    retry_delay_seconds=60,
    task_run_name="get_data-{endpoint}"
)
def extract_data(endpoint: str):
    data_url = f"{BASE_URL}/{endpoint}"
    response = requests.get(data_url)
    response.raise_for_status()
    data = response.json()["data"]
    return data

@task(
    task_run_name="transform_data"
)
def transform_data(data: list):
    df = pd.json_normalize(data)
    return df

@task(
    task_run_name="load_data-{endpoint_name}"
)
def load_data(df: pd.DataFrame, endpoint_name: str):
    scheduled_run = flow_run.scheduled_start_time.strftime("%Y-%m-%dT%H:%M")
    os.makedirs(f"data/{endpoint_name}", exist_ok=True)
    df.to_csv(f"data/{endpoint_name}/data_{scheduled_run}.csv")

@flow
def etl_parametrizable(endpoints: list):
    for endpoint in endpoints:
        data = extract_data(endpoint)
        df = transform_data(data)
        load_data(df, endpoint)

if __name__ == "__main__":
  etl_parametrizable.serve(name="ETL-Parametrizable")

Writing scripts/etl_parametrizable.py


In [ ]:
!python scripts/etl_parametrizable.py

Your flow 'etl-parametrizable' is being served and polling for scheduled runs!

To trigger a run for this flow, use the following command:

        $ prefect deployment run 'etl-parametrizable/ETL-Parametrizable'

You can also run your flow via the Prefect UI: https://app.prefect.cloud/account/4f464ca9-2008-4694-bae1-d3dad8f2df02/workspace/666ec2f7-4658-47d3-9d17-ebdeb0399ca6/deployments/deployment/1e3b25e6-adcb-4066-a2ef-3e885e4d0ad3

00:28:28.814 | INFO    | prefect.flow_runs.runner - Runner 'ETL-Parametrizable' submitting flow run '083c31b9-19ba-4053-8a27-dbb0544c3dba'
00:28:28.955 | INFO    | prefect.flow_runs.runner - Opening process...
00:28:28.973 | INFO    | prefect.flow_runs.runner - Completed submission of flow run '083c31b9-19ba-4053-8a27-dbb0544c3dba'
00:28:31.798 | INFO    | Flow run 'meaty-skylark' - Downloading flow code from storage at '.'
00:28:32.353 | INFO    | Flow run 'meaty-skylark' - Beginning flow run 'meaty-skylark' for flow 'etl-parametrizable'
00:28:32.355 | 

## ETL Parametrizable v2
La versión puede considerarse secuencial.
Se aplica todo el proceso ETL para el 1er endpoint, una vez terminado continúa con el mismo flujo ETL para el siguiente y así sucesivamente

In [ ]:
%%writefile scripts/etl_parametrizable_v2.py

import os
import requests
import pandas as pd
from prefect import task, flow
from prefect.runtime import flow_run

BASE_URL = "https://api.luchtmeetnet.nl/open_api"

@task(
    retries=3,
    retry_delay_seconds=60,
    )
def extract_data(endpoint: str):
    data_url = f"{BASE_URL}/{endpoint}"
    response = requests.get(data_url)
    response.raise_for_status()
    data = response.json()["data"]
    return data

@task
def transform_data(data: list):
    df = pd.json_normalize(data)
    return df

@task
def load_data(df: pd.DataFrame, endpoint_name: str):
    scheduled_run = flow_run.scheduled_start_time.strftime("%Y-%m-%dT%H:%M")
    os.makedirs(f"data/{endpoint_name}", exist_ok=True)
    df.to_csv(f"data/{endpoint_name}/data_{scheduled_run}.csv")

@task(
    task_run_name="etl-{endpoint}"
)
def etl_parametrizable(endpoint: str):
    data = extract_data(endpoint)
    df = transform_data(data)
    load_data(df, endpoint)

@flow
def etl_parametrizable_v2(endpoints: list):
    etl_parametrizable.map(endpoints)

if __name__ == "__main__":
  etl_parametrizable_v2.serve(name="ETL-Parametrizable-v2")

Writing scripts/etl_parametrizable_v2.py


In [ ]:
!python scripts/etl_parametrizable_v2.py

Your flow 'etl-parametrizable-v2' is being served and polling for scheduled runs!

To trigger a run for this flow, use the following command:

        $ prefect deployment run 'etl-parametrizable-v2/ETL-Parametrizable-v2'

You can also run your flow via the Prefect UI: https://app.prefect.cloud/account/4f464ca9-2008-4694-bae1-d3dad8f2df02/workspace/666ec2f7-4658-47d3-9d17-ebdeb0399ca6/deployments/deployment/601910eb-9220-4ea0-ad26-87e846418d1c

00:34:18.480 | INFO    | prefect.flow_runs.runner - Runner 'ETL-Parametrizable-v2' submitting flow run '54098f69-b1a5-42cd-9eb7-765a7ae65c07'
00:34:18.641 | INFO    | prefect.flow_runs.runner - Opening process...
00:34:18.658 | INFO    | prefect.flow_runs.runner - Completed submission of flow run '54098f69-b1a5-42cd-9eb7-765a7ae65c07'
00:34:21.538 | INFO    | Flow run 'primitive-tench' - Downloading flow code from storage at '.'
00:34:22.116 | INFO    | Flow run 'primitive-tench' - Beginning flow run 'primitive-tench' for flow 'etl-parametrizabl

## Incremental

In [ ]:
%%writefile scripts/etl_mediciones.py

import os
import requests
import pandas as pd
from datetime import timedelta
from prefect import task, flow
from prefect.runtime import flow_run

BASE_URL = "https://api.luchtmeetnet.nl/open_api"

start_date = flow_run.scheduled_start_time - timedelta(minutes=5)
start_date = start_date.strftime("%Y-%m-%dT%H:%M:%SZ")
end_date = flow_run.scheduled_start_time.strftime("%Y-%m-%dT%H:%M:%SZ")
#end_date = start_date.strftime("%Y-%m-%dT%H:59:59Z")
#start_date = start_date.strftime("%Y-%m-%dT%H:00:00Z")

@task(
      retries=3,
      retry_delay_seconds=60
)
def get_data(
    endpoint: str,
    start_date: str,
    end_date: str
    ):
    data_url = f"{BASE_URL}/{endpoint}"
    response = requests.get(
        data_url,
        params={
            "start": start_date,
            "end": end_date
            })
    response.raise_for_status()
    return response.json()["data"]

@task
def transform_and_load(data: list, start_date: str):
    df = pd.json_normalize(data)
    os.makedirs(f"data/mediciones/", exist_ok=True)
    df.to_csv(
        f"data/mediciones/data_{start_date}.csv")

@flow(
    name="mediciones-etl-v2"
)
def etl(start_date, end_date):
    data = get_data("measurements", start_date=start_date, end_date=end_date)
    transform_and_load(data, start_date)

if __name__ == "__main__":
    etl.serve(
        name="mediciones-etl-v2",
        cron="*/5 * * * *",
        parameters={
            "start_date": start_date,
            "end_date": end_date
            }
    )

Writing scripts/etl_mediciones.py


In [ ]:
!python scripts/etl_mediciones.py

Your flow 'mediciones-etl-v2' is being served and polling for scheduled runs!

To trigger a run for this flow, use the following command:

        $ prefect deployment run 'mediciones-etl-v2/mediciones-etl-v2'

You can also run your flow via the Prefect UI: https://app.prefect.cloud/account/4f464ca9-2008-4694-bae1-d3dad8f2df02/workspace/666ec2f7-4658-47d3-9d17-ebdeb0399ca6/deployments/deployment/ed463e16-001d-4abe-adf3-8aa911328476

00:39:59.049 | INFO    | prefect.flow_runs.runner - Runner 'mediciones-etl-v2' submitting flow run '4e61c943-37c8-45c2-899a-b2258af90f2a'
00:39:59.207 | INFO    | prefect.flow_runs.runner - Opening process...
00:39:59.230 | INFO    | prefect.flow_runs.runner - Completed submission of flow run '4e61c943-37c8-45c2-899a-b2258af90f2a'
00:40:02.840 | INFO    | Flow run 'skinny-stallion' - Downloading flow code from storage at '.'
00:40:04.009 | INFO    | Flow run 'skinny-stallion' - Beginning flow run 'skinny-stallion' for flow 'mediciones-etl-v2'
00:40:04.011 |

## Otro ejemplo de dependencias

In [ ]:
%%writefile get_station_by_org.py

import requests
from prefect import task, Flow

BASE_URL = "https://api.luchtmeetnet.nl/open_api"

@task(
      task_run_name="get_station_data-organisation-{organisation_id}",
      retries=3,
      retry_delay_seconds=60
)
def get_station_data(organisation_id: int):
    station_url = f"{BASE_URL}/stations"
    response = requests.get(
        station_url,
        params={
            "organisation_id": organisation_id
            })
    response.raise_for_status()
    return response.json()

@Flow
def get_all_stations():
    org_ids = [1, 2, 4, 5, 6, 7, 8]
    get_station_result = get_station_data.map(org_ids)
    for result in get_station_result:
        result.wait()

if __name__ == "__main__":
    get_all_stations.serve(
        name="get-station-by-org",
        )

In [ ]:
!python get_station_by_org.py

In [ ]:
%%writefile scripts/get_station_by_org_v2.py

import requests
from prefect import task, flow

BASE_URL = "https://api.luchtmeetnet.nl/open_api"

@task
def get_organisations_data():
    """
    Obtener ids de las organizaciones
    """
    organisation_url = f"{BASE_URL}/organisations"
    response = requests.get(organisation_url)
    response.raise_for_status()
    organisations = response.json()["data"]
    org_ids = [org["id"] for org in organisations]
    return org_ids

@task(
      task_run_name="get_station_data-organisation-{organisation_id}",
      retries=3,
      retry_delay_seconds=60
)
def get_station_data(organisation_id: int):
    station_url = f"{BASE_URL}/stations"
    response = requests.get(station_url,
                params={
                    "organisation_id": organisation_id
                    })
    response.raise_for_status()
    return response.json()


@flow(
      name="Get all stations v2",
      description="Get all stations from all organisations"
 )
def get_all_stations():
    org_ids = get_organisations_data()
    get_station_result = get_station_data.map(org_ids)
    for result in get_station_result:
        result.wait()

if __name__ == "__main__":
    get_all_stations.serve(
        name="get-station-by-org-v2",
        )

In [ ]:
!python scripts/get_station_by_org_v2.py